In [ ]:
from sklearn import datasets
from sklearn.model_selection import train_test_split
# ??sklearn???mnist??
digits = datasets.load_digits()
# ?????1797?????????,?????8*8
# ????0~10,????????10????
images = digits.images
targets = digits.target
print(("dataset shape is: "), images.shape)
# ??????????????(20%)
X_train,X_test,y_train,y_test = train_test_split(images , targets , test_size=0.2 , random_state=0)
num_training = 1137
num_validation = 300
num_test = y_test.shape[0]
# ??????????????
mask = list(range(num_training, num_training + num_validation))
X_val = X_train[mask]
y_val = y_train[mask]
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]
print("the number of train: ", num_training)
print("the number of test: ", num_test)
print("the number of validation: ", num_validation)
# ?????8*8????????64*1???
X_train = X_train.reshape(num_training, -1)
X_val = X_val.reshape(num_validation, -1)
X_test = X_test.reshape(num_test, -1)
print("training data shape: ", X_train.shape)
print("validation data shape: ", X_val.shape)
print("test data shape: ", X_test.shape)







In [ ]:
# ?????????
# ????
input_size = 64
hidden_size = 30
num_classes = 10
# ?????????,???????,??loss,??,?????????????network???
import numpy as np
import matplotlib.pyplot as plt
class network(object):
    # ?????,?W,b?????params????
    # W?????,b?????
    def __init__(self, input_size, hidden_size, output_size, std=1e-4):
        self.params = {}
        self.params['W1'] = std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)
    # ??????,??????????????
    def loss(self, X, y=None, reg=0.0):
        # ???????????
        W1 = self.params['W1']
        b1 = self.params['b1']
        W2 = self.params['W2']
        b2 = self.params['b2']
        N, D = X.shape
        # ????
        # hidden???
        hidden = np.dot(X, W1) + b1
        # relu:max(0, x)
        hidden = np.maximum(0,hidden)
        # ???y
        y2 = np.dot(hidden, W2) + b2
        #if y == None:
          #  return y2
        # loss ??
        loss = None
        loss = -y2[range(N), y].sum() + np.log(np.exp(y2).sum(axis = 1)).sum()
        loss = loss / N + 0.5 * reg * (np.sum(W1 * W1) + np.sum(W2 * W2))
        # ????
        # ??????grads???,????????????
        grads = {}
        # ?????,???dscore
        # ??y2???
        exp = np.exp(y2)
        # ????????,???????????????
        dscore = exp / exp.sum(axis = 1, keepdims = True)
        # ?label(?y)??????1
        dscore[range(N),y] -= 1
        # ?????????????
        dscore = dscore/N 
        grads['b2'] = np.sum(dscore, axis=0)
        grads['W2'] = np.dot(hidden.T, dscore)+ reg * W2
        # dhidden
        dhidden = np.dot(dscore, W2.T)
        # ????relu????,????XW1 + b1 <0???dihidden???0
        dhidden[(np.dot(X, W1)+b1)<0] = 0
        grads['b1'] = np.sum(dhidden, axis=0)
        grads['W1'] = np.dot(X.T, dhidden) + reg * W1
        return loss, grads
    # ??????,?????????,?????????
    def train(self, X, y, X_val, y_val,
            learning_rate=1e-3, learning_rate_decay=0.95,
            reg=5e-6, num_iters=100,
            batch_size=200, verbose=False):
        # ???????????,????????batch????epoch???????
        num_train = X.shape[0]
        iterations_per_epoch = max(num_train / batch_size, 1)

        # ???????????????
        # ?????????loss??????????????????
        loss_history = []
        train_acc_history = []
        val_acc_history = []
        # ??numz_iters?,?????????batch?????
        for it in range(num_iters):
            X_batch = None
            y_batch = None
            indices = np.random.choice(num_train, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            # ????batch???????loss ?grad
            loss, grads = self.loss(X_batch, y=y_batch, reg=reg)
            # ???????????
            loss_history.append(loss)
            self.params['W1'] -=  learning_rate * grads['W1']
            self.params['b1'] -=  learning_rate * grads['b1']
            self.params['W2'] -=  learning_rate * grads['W2']
            self.params['b2'] -=  learning_rate * grads['b2']
            # ?????????????,????????????loss
            if verbose and it % 100 == 0:
                print('iteration %d / %d: loss %f' % (it, num_iters, loss))

           # Every epoch, check train and val accuracy and decay learning rate.
            if it % iterations_per_epoch == 0:
                # Check accuracy
                train_acc = (self.predict(X_batch) == y_batch).mean()
                val_acc = (self.predict(X_val) == y_val).mean()
                train_acc_history.append(train_acc)
                val_acc_history.append(val_acc)

                # ??epoch??,???????
                learning_rate *= learning_rate_decay

        return {
            'loss_history': loss_history,
            'train_acc_history': train_acc_history,
            'val_acc_history': val_acc_history,
        }

    def predict(self, X):
    
        y_pred = None
        W1, b1 = self.params['W1'], self.params['b1']
        W2, b2 = self.params['W2'], self.params['b2']

        hidden_layer = np.maximum(0, np.dot(X, W1) + b1)
        scores = np.dot(hidden_layer, W2) + b2
        y_pred = np.argmax(scores, axis = 1)
        return y_pred




        
        

In [ ]:
net = network(input_size, hidden_size, num_classes)
stats = net.train(X_train, y_train, X_val, y_val,
            num_iters=5000, batch_size=200,
            learning_rate=0.01, learning_rate_decay=0.95,
            reg=0.25, verbose=True)
val_acc = (net.predict(X_val) == y_val).mean()
print('Validation accuracy: ', val_acc)

test_acc = (net.predict(X_test) == y_test).mean()
print('test accuracy: ', test_acc)


